# WOT Interpolation Visualization (Flow-Sinkhorn Graph)

Goal: visualize endpoint-based graph interpolation and contrast with true intermediate WOT snapshots.

This notebook:
1. Loads and preprocesses WOT tutorial data.
2. Subsamples up to `n0` cells per time point.
3. Builds a global k-NN graph in PCA space.
4. Computes shortest-path geodesics on graph.
5. Builds endpoint coupling (`mu_1 -> mu_T`) and interpolates mass along shortest paths.
6. Compares interpolated mass vs ground-truth day snapshots.


In [ ]:
# Optional first-time setup
# !pip install -e .
# !pip install -r singlecell/requirements.txt


In [ ]:
# Compute backend configuration
USE_GPU = False  # set True to use GPU when available
DEVICE = 'cuda' if USE_GPU else 'cpu'
USE_TORCH = True  # torch sparse solver backend


In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from benchmarks.wot_benchmark import (
    BenchmarkConfig,
    download_wot_tutorial_data,
    extract_zip,
    prepare_wot_data,
    interpolate_from_endpoint_coupling,
)


In [ ]:
DATA_ROOT = Path("data/wot")
DO_DOWNLOAD = False
DO_EXTRACT = False

if DO_DOWNLOAD:
    download_wot_tutorial_data(DATA_ROOT)
if DO_EXTRACT:
    extract_zip(DATA_ROOT / "data.zip", DATA_ROOT)

cfg = BenchmarkConfig(
    data_root=DATA_ROOT,
    random_state=0,
    n_cells_per_time=200,
    pca_components=30,
    knn_k=4,
    epsilon=0.05,
    niter=400,
    use_torch=USE_TORCH,
    max_timepoints=12,
)

prepared = prepare_wot_data(cfg)
print("n_cells:", prepared["n_cells"], "n_days:", len(prepared["days"]))
print("days:", prepared["days"])


In [ ]:
interp = interpolate_from_endpoint_coupling(
    prepared,
    epsilon=0.05,
    sinkhorn_niter=400,
    mass_threshold=1e-6,
)

tv_df = pd.DataFrame({
    "day": list(interp["tv_by_day"].keys()),
    "tv": list(interp["tv_by_day"].values()),
}).sort_values("day")

tv_df.head(), tv_df.tail()


In [ ]:
# TV distance curve vs day (lower is better)
plt.figure(figsize=(7, 3.5))
plt.plot(tv_df["day"], tv_df["tv"], marker="o")
plt.xlabel("day")
plt.ylabel("TV(interp, ground truth)")
plt.title("Interpolation quality over time")
plt.grid(alpha=0.25)
plt.tight_layout()
plt.show()


In [ ]:
# Side-by-side visual comparison on PCA coordinates
emb = prepared["embedding"]
days = prepared["days"]
interp_mass = interp["interp_mass_by_day"]
gt_mass = interp["ground_truth_mass_by_day"]

n_show = min(6, len(days))
show_days = np.linspace(0, len(days)-1, n_show, dtype=int)
show_days = [float(days[i]) for i in show_days]

fig, axes = plt.subplots(n_show, 2, figsize=(10, 3.2*n_show), sharex=True, sharey=True)
if n_show == 1:
    axes = np.array([axes])

for r, d in enumerate(show_days):
    m_pred = interp_mass[d]
    m_gt = gt_mass[d]

    ax = axes[r, 0]
    s = 12 + 800 * m_pred
    ax.scatter(emb[:, 0], emb[:, 1], s=s, c=m_pred, cmap="magma", alpha=0.8)
    ax.set_title(f"Interpolated mass (day={d:g})")
    ax.set_xlabel("PC1")
    ax.set_ylabel("PC2")

    ax = axes[r, 1]
    s = 12 + 800 * m_gt
    ax.scatter(emb[:, 0], emb[:, 1], s=s, c=m_gt, cmap="viridis", alpha=0.8)
    ax.set_title(f"Ground truth snapshot (day={d:g})")
    ax.set_xlabel("PC1")
    ax.set_ylabel("PC2")

plt.tight_layout()
plt.show()


In [ ]:
# Optional: save summary table
out = Path("singlecell/results/interp_tv_by_day.csv")
out.parent.mkdir(parents=True, exist_ok=True)
tv_df.to_csv(out, index=False)
out
